# 01. Exploratory Data Analysis (EDA) — Amazon ML Challenge
**Phase 1: Local Entity Resolution Pipeline**

This notebook performs comprehensive exploratory data analysis on the entity matching dataset:
- 
- 
- 
- 

## 1. Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append("..")
from src.data_loader import load_raw_datasets, get_dataset_info
from src.preprocessing import preprocess_dataframe

sns.set_theme(style="whitegrid")

## 2. Load Data

In [ ]:
DATA_DIR = "../data"
s1, s2, s3, gt = load_raw_datasets(data_dir=DATA_DIR)
print(f"Source 1 shape: {s1.shape}")
print(f"Source 2 shape: {s2.shape}")
print(f"Source 3 shape: {s3.shape}")
print(f"Ground Truth shape: {gt.shape}")

## 3. Dataset Overview

In [ ]:
info = get_dataset_info(data_dir=DATA_DIR)
overview_df = pd.DataFrame.from_dict(info, orient="index")
display(overview_df)

## 4. Missing-Value Analysis

In [ ]:
missing_records = []
for name, df in [("Source 1", s1), ("Source 2", s2), ("Source 3", s3)]:
    for col in df.columns:
        cnt = df[col].isnull().sum()
        pct = (cnt / len(df)) * 100
        missing_records.append({"Source": name, "Column": col, "Missing Count": cnt, "Missing %": round(pct, 4)})

for col in gt.columns:
    cnt = gt[col].isnull().sum()
    pct = (cnt / len(gt)) * 100
    missing_records.append({"Source": "Ground Truth", "Column": col, "Missing Count": cnt, "Missing %": round(pct, 4)})

missing_df = pd.DataFrame(missing_records)
display(missing_df)

## 5. Duplicate Analysis

In [ ]:
dup_records = []
for name, df in [("Source 1", s1), ("Source 2", s2), ("Source 3", s3)]:
    dup_ids = df["entity_id"].duplicated().sum()
    dup_rows = df.duplicated().sum()
    dup_records.append({"Source": name, "Duplicate Entity IDs": dup_ids, "Exact Duplicate Rows": dup_rows})
display(pd.DataFrame(dup_records))

## 6. Unique-Value Analysis

In [ ]:
cardinality = []
for name, df in [("Source 1", s1), ("Source 2", s2), ("Source 3", s3)]:
    cardinality.append({
        "Source": name,
        "Unique Entity IDs": df["entity_id"].nunique(),
        "Unique Business Names": df["business_name"].nunique(),
        "Unique Addresses": df["business_address"].nunique(),
        "Unique Countries": df["country"].nunique()
    })
display(pd.DataFrame(cardinality))

## 7. Business-Name Analysis

In [ ]:
print("Top 5 most frequent business names in Source 1:")
display(s1["business_name"].value_counts().head(5))
print("
Top 5 most frequent business names in Source 2:")
display(s2["business_name"].value_counts().head(5))

## 8. Address Analysis

In [ ]:
print("Top 5 most frequent business addresses in Source 1:")
display(s1["business_address"].value_counts().head(5))
print("
Missing addresses percentage across sources:")
print(f"Source 2 missing addresses: {s2['business_address'].isnull().mean()*100:.2f}%")
print(f"Source 3 missing addresses: {s3['business_address'].isnull().mean()*100:.2f}%")

## 9. Country Analysis

In [ ]:
country_df = pd.DataFrame({
    "Source 1": s1["country"].value_counts(),
    "Source 2": s2["country"].value_counts(),
    "Source 3": s3["country"].value_counts()
})
display(country_df)
country_df.plot(kind="bar", figsize=(8, 5))
plt.title("Country Distribution across Data Sources")
plt.ylabel("Record Count")
plt.tight_layout()
plt.savefig("../outputs/reports/country_distribution.png")
plt.show()

## 10. Ground-Truth Analysis

In [ ]:
def parse_gt_row(val):
    if pd.isna(val) or not str(val).strip():
        return [], []
    ids = [i.strip() for i in str(val).split(",") if i.strip()]
    return [i for i in ids if i.startswith("S2-")], [i for i in ids if i.startswith("S3-")]

gt_parsed = gt["matched_entity_ids"].apply(parse_gt_row)
total_matches = gt_parsed.apply(lambda x: len(x[0]) + len(x[1]))
s2_matches = gt_parsed.apply(lambda x: len(x[0]))
s3_matches = gt_parsed.apply(lambda x: len(x[1]))

print("Match Count Distribution:")
display(total_matches.value_counts().sort_index().to_frame("Source 1 Entities"))

plt.figure(figsize=(9, 4))
sns.countplot(x=total_matches[total_matches <= 10])
plt.title("Distribution of Matches per Source 1 Entity")
plt.xlabel("Number of Matched Entities (S2 + S3)")
plt.ylabel("Count of S1 Entities")
plt.tight_layout()
plt.savefig("../outputs/reports/ground_truth_matches_distribution.png")
plt.show()

## 11. Initial Entity-Resolution Observations

1. **Ground Truth Multiplicity**: Ground truth provides 1-to-many matches. An entity in Source 1 can match multiple entities across Source 2 and Source 3 (up to 11 matches).
2. **Country Blocking Potential**: Datasets are partitioned into US (~60%) and India (~40%). Partitioning candidate generation by country avoids cross-country false positives.
3. **Missing Address Ratio**: ~3.35% of addresses in S2 and S3 are missing. Fallback logic for missing addresses is necessary during string similarity calculation.
4. **Zero Duplicates**: Source files contain zero duplicate  values and zero exact duplicate rows.

## 12. Conclusions

- The dataset fits within ~1.63 GB RAM.
- Candidate blocking is mandatory to avoid .2M 	imes 5.0M$ Cartesian explosions.
- Next step: Implement string normalization () and blocking strategies ().